In [1]:
pip install openmeteo-requests requests-cache retry-requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
LATITUDE = 51.5074
LONGITUDE = -0.1278

In [3]:
import requests
import pandas as pd

url = (
    "https://archive-api.open-meteo.com/v1/archive"
)

params = {
    "latitude": 51.5074,
    "longitude": -0.1278,
    "start_date": "2001-01-01",
    "end_date": "2025-12-31",
    "daily": [
        "temperature_2m_mean",
        "temperature_2m_max",
        "temperature_2m_min",
        "precipitation_sum",
        "wind_speed_10m_max"
    ],
    "timezone": "Europe/London"
}

response = requests.get(url, params=params)

weather_json = response.json()

print(weather_json.keys())

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily'])


In [4]:
weather_df = pd.DataFrame({
    "SETTLEMENT_DATE":
        weather_json["daily"]["time"],

    "temperature_mean":
        weather_json["daily"]["temperature_2m_mean"],

    "temperature_max":
        weather_json["daily"]["temperature_2m_max"],

    "temperature_min":
        weather_json["daily"]["temperature_2m_min"],

    "rainfall":
        weather_json["daily"]["precipitation_sum"],

    "wind_speed":
        weather_json["daily"]["wind_speed_10m_max"]
})

weather_df.head()

,SETTLEMENT_DATE,temperature_mean,temperature_max,temperature_min,rainfall,wind_speed
0,2001-01-01,8.3,10.5,3.3,6.3,30.8
1,2001-01-02,8.6,9.7,6.7,2.6,35.1
2,2001-01-03,5.9,9.4,1.8,3.2,26.3
3,2001-01-04,8.1,9.8,5.8,12.9,24.8
4,2001-01-05,6.4,7.2,5.3,5.8,14.1


In [5]:
weather_df["SETTLEMENT_DATE"] = pd.to_datetime(
    weather_df["SETTLEMENT_DATE"]
)

In [6]:
weather_df.to_parquet(
    "../data/processed/weather_features.parquet",
    index=False
)

print(weather_df.shape)

(9131, 6)


In [8]:
import pandas as pd

daily_df = pd.read_parquet(
    "../data/processed/daily_energy_holidays.parquet"
)

print(daily_df.shape)

daily_df.head()

(9264, 7)


,SETTLEMENT_DATE,ND,is_holiday,holiday_name,is_christmas,is_new_year,is_easter
0,2001-01-01,35616.520833,1,New Year's Day,0,1,0
1,2001-01-02,47423.416667,0,NaN,0,0,0
2,2001-01-03,47082.916667,0,NaN,0,0,0
3,2001-01-04,34527.708333,0,NaN,0,0,0
4,2001-01-05,39791.687500,0,NaN,0,0,0


In [9]:
weather_df = pd.read_parquet(
    "../data/processed/weather_features.parquet"
)

print(weather_df.shape)

weather_df.head()

(9131, 6)


,SETTLEMENT_DATE,temperature_mean,temperature_max,temperature_min,rainfall,wind_speed
0,2001-01-01,8.3,10.5,3.3,6.3,30.8
1,2001-01-02,8.6,9.7,6.7,2.6,35.1
2,2001-01-03,5.9,9.4,1.8,3.2,26.3
3,2001-01-04,8.1,9.8,5.8,12.9,24.8
4,2001-01-05,6.4,7.2,5.3,5.8,14.1


In [10]:
merged_df = daily_df.merge(
    weather_df,
    on="SETTLEMENT_DATE",
    how="left"
)

print(merged_df.shape)

(9264, 12)


In [11]:
merged_df.isnull().sum()

SETTLEMENT_DATE        0
ND                     0
is_holiday             0
holiday_name        9085
is_christmas           0
is_new_year            0
is_easter              0
temperature_mean     133
temperature_max      133
temperature_min      133
rainfall             133
wind_speed           133
dtype: int64

In [12]:
merged_df[
    merged_df["temperature_mean"].isna()
][["SETTLEMENT_DATE"]].head(20)

,SETTLEMENT_DATE
9131,2026-01-01
9132,2026-01-02
9133,2026-01-03
9134,2026-01-04
9135,2026-01-05
9136,2026-01-13
9137,2026-01-14
9138,2026-01-15
9139,2026-01-16
9140,2026-01-17


In [13]:
merged_df[
    merged_df["temperature_mean"].isna()
][["SETTLEMENT_DATE"]].tail(20)

,SETTLEMENT_DATE
9244,2026-09-01
9245,2026-09-02
9246,2026-09-03
9247,2026-09-04
9248,2026-09-05
9249,2026-10-01
9250,2026-10-02
9251,2026-10-03
9252,2026-10-04
9253,2026-10-05


In [14]:
merged_df = merged_df.dropna(
    subset=[
        "temperature_mean",
        "temperature_max",
        "temperature_min",
        "rainfall",
        "wind_speed"
    ]
)

print(merged_df.shape)

(9131, 12)


In [15]:
merged_df = merged_df.drop(
    columns=["holiday_name"]
)

In [16]:
merged_df.to_parquet(
    "../data/processed/energy_weather_holiday.parquet",
    index=False
)

print("Version 2 dataset saved")

Version 2 dataset saved


In [17]:
FEATURES = [
    "lag_1",
    "lag_7",
    "lag_30",

    "rolling_mean_7",
    "rolling_mean_30",

    "year",
    "month",
    "quarter",
    "day_of_week",
    "day_of_year",
    "is_weekend",

    "is_holiday",
    "is_christmas",
    "is_new_year",
    "is_easter",

    "temperature_mean",
    "temperature_max",
    "temperature_min",

    "rainfall",
    "wind_speed",

    "temp_lag_1",
    "rainfall_lag_1",
    "wind_lag_1"
]

In [19]:
missing_weather = merged_df[
    merged_df["temperature_mean"].isna()
]

print(missing_weather.shape)

missing_weather[
    ["SETTLEMENT_DATE"]
].head(20)

(0, 11)


,SETTLEMENT_DATE


In [20]:
weather_df["SETTLEMENT_DATE"].min()

weather_df["SETTLEMENT_DATE"].max()

merged_df["SETTLEMENT_DATE"].min()

merged_df["SETTLEMENT_DATE"].max()

Timestamp('2025-12-31 00:00:00')

In [21]:
import os

print(os.listdir("../data/processed"))

['daily_energy.parquet', 'daily_energy_holidays.parquet', 'energy_features.parquet', 'energy_weather_holiday.parquet', 'uk_energy_master.csv', 'uk_energy_master.parquet', 'weather_features.parquet']


In [22]:
import pandas as pd

df = pd.read_parquet(
    "../data/processed/energy_weather_holiday.parquet"
)

print(df.shape)

print(df.columns.tolist())

(9131, 11)
['SETTLEMENT_DATE', 'ND', 'is_holiday', 'is_christmas', 'is_new_year', 'is_easter', 'temperature_mean', 'temperature_max', 'temperature_min', 'rainfall', 'wind_speed']


In [23]:
df.head()

,SETTLEMENT_DATE,ND,is_holiday,is_christmas,is_new_year,is_easter,temperature_mean,temperature_max,temperature_min,rainfall,wind_speed
0,2001-01-01,35616.520833,1,0,1,0,8.3,10.5,3.3,6.3,30.8
1,2001-01-02,47423.416667,0,0,0,0,8.6,9.7,6.7,2.6,35.1
2,2001-01-03,47082.916667,0,0,0,0,5.9,9.4,1.8,3.2,26.3
3,2001-01-04,34527.708333,0,0,0,0,8.1,9.8,5.8,12.9,24.8
4,2001-01-05,39791.687500,0,0,0,0,6.4,7.2,5.3,5.8,14.1


In [24]:
df.tail()

,SETTLEMENT_DATE,ND,is_holiday,is_christmas,is_new_year,is_easter,temperature_mean,temperature_max,temperature_min,rainfall,wind_speed
9126,2025-12-27,27954.104167,0,0,0,0,5.1,7.4,2.5,0.0,22.2
9127,2025-12-28,28898.979167,0,0,0,0,6.5,7.1,4.5,0.0,14.8
9128,2025-12-29,30724.208333,0,0,0,0,4.9,6.1,3.0,0.0,11.9
9129,2025-12-30,30899.979167,0,0,0,0,4.5,6.1,1.9,0.0,12.2
9130,2025-12-31,30377.208333,0,0,0,0,0.7,2.7,-1.3,0.0,11.4


In [25]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9131 entries, 0 to 9130
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   SETTLEMENT_DATE   9131 non-null   datetime64[us]
 1   ND                9131 non-null   float64       
 2   is_holiday        9131 non-null   int64         
 3   is_christmas      9131 non-null   int64         
 4   is_new_year       9131 non-null   int64         
 5   is_easter         9131 non-null   int64         
 6   temperature_mean  9131 non-null   float64       
 7   temperature_max   9131 non-null   float64       
 8   temperature_min   9131 non-null   float64       
 9   rainfall          9131 non-null   float64       
 10  wind_speed        9131 non-null   float64       
dtypes: datetime64[us](1), float64(6), int64(4)
memory usage: 784.8 KB
